In [ ]:
# ============================================================
# 1. DATASET PREPARATION
# ============================================================

import sys
import subprocess
import importlib.util
import os
import random
from pathlib import Path

packages = {
    "kagglehub": "kagglehub",
    "cv2": "opencv-python-headless",
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib"
}

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

import cv2
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from PIL import Image

from torch.utils.data import Dataset, DataLoader

from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# Settings

SEED = 42

DATASET_ID = "nodoubttome/skin-cancer9-classesisic"

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.0001
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# Reproducibility

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Download dataset

print("\nDownloading dataset...")

dataset_root = Path(
    kagglehub.dataset_download(DATASET_ID)
)

print("Dataset location:")
print(dataset_root)


# Find Train/Test folders

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


def has_images(folder):

    return any(
        file.is_file()
        and file.suffix.lower() in IMAGE_EXTENSIONS
        for file in Path(folder).iterdir()
    )


def find_split(root, split_name):

    for current, dirs, files in os.walk(root):

        folder = Path(current)

        if folder.name.lower() == split_name.lower():

            class_folders = [
                x for x in folder.iterdir()
                if x.is_dir() and has_images(x)
            ]

            if len(class_folders) >= 2:
                return folder

    return None


TRAIN_DIR = find_split(
    dataset_root,
    "Train"
)

TEST_DIR = find_split(
    dataset_root,
    "Test"
)


if TRAIN_DIR is None:

    raise FileNotFoundError(
        "Train folder could not be found."
    )


print("\nTrain folder:")
print(TRAIN_DIR)

print("\nTest folder:")
print(TEST_DIR)


# Load training dataset

base_dataset = datasets.ImageFolder(
    TRAIN_DIR
)

classes = base_dataset.classes

NUM_CLASSES = len(classes)


print("\nClasses:")

for i, class_name in enumerate(classes):
    print(i, "-", class_name)


print("\nTotal training images:",
      len(base_dataset))


# Fixed 85/15 train-validation split

all_indices = np.arange(
    len(base_dataset)
)

all_labels = np.array(
    base_dataset.targets
)


train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=0.15,
    random_state=SEED,
    stratify=all_labels
)


print("\nTraining samples:",
      len(train_indices))

print("Validation samples:",
      len(validation_indices))


# Test dataset

if TEST_DIR is not None:

    test_dataset = datasets.ImageFolder(
        TEST_DIR
    )

    if test_dataset.class_to_idx != base_dataset.class_to_idx:

        raise ValueError(
            "Train and Test class mappings are different."
        )

    print("Test samples:",
          len(test_dataset))

else:

    test_dataset = None

    print("No separate test folder found.")


# Transformations

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])


evaluation_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

Device: cuda

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Dataset location:
/kaggle/input/skin-cancer9-classesisic

Train folder:
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train

Test folder:
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test

Classes:
0 - actinic keratosis
1 - basal cell carcinoma
2 - dermatofibroma
3 - melanoma
4 - nevus
5 - pigmented benign keratosis
6 - seborrheic keratosis
7 - squamous cell carcinoma
8 - vascular lesion

Total training images: 2239

Training samples: 1903
Validation samples: 336
Test samples: 118


In [ ]:
# ============================================================
# 2. MODEL LOADING
# ============================================================

MODELS = [
    "EfficientNet-B0",
    "DenseNet121",
    "ResNet101"
]


def load_model(model_name):

    if model_name == "EfficientNet-B0":

        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )

        model.classifier[-1] = nn.Linear(
            model.classifier[-1].in_features,
            NUM_CLASSES
        )


    elif model_name == "DenseNet121":

        model = models.densenet121(
            weights=models.DenseNet121_Weights.DEFAULT
        )

        model.classifier = nn.Linear(
            model.classifier.in_features,
            NUM_CLASSES
        )


    elif model_name == "ResNet101":

        model = models.resnet101(
            weights=models.ResNet101_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES
        )


    else:

        raise ValueError(
            "Unknown model: " + model_name
        )


    return model.to(DEVICE)


print("Models selected:")

for model_name in MODELS:
    print("-", model_name)

Models selected:
- EfficientNet-B0
- DenseNet121
- ResNet101


In [ ]:
# ============================================================
# 3. BASELINE EXPERIMENT
# ============================================================

# Baseline uses the original images
# without applying any image filter.

BASELINE_NAME = "No Filter"

print("\nBaseline experiment:")
print("Original images - No Filter")


Baseline experiment:
Original images - No Filter


In [ ]:
# ============================================================
# 4. IMAGE FILTERING
# ============================================================

FILTERS = [
    "No Filter",
    "Average",
    "Gaussian",
    "Median",
    "Sharpening",
    "Sobel"
]


def apply_filter(image, filter_name):

    # Original image
    if filter_name == "No Filter":

        return image


    image_array = np.array(
        image
    )


    # Average / Mean Filter

    if filter_name == "Average":

        filtered = cv2.blur(
            image_array,
            (5, 5)
        )


    # Gaussian Filter

    elif filter_name == "Gaussian":

        filtered = cv2.GaussianBlur(
            image_array,
            (5, 5),
            0
        )


    # Median Filter

    elif filter_name == "Median":

        filtered = cv2.medianBlur(
            image_array,
            5
        )


    # Sharpening Filter

    elif filter_name == "Sharpening":

        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ], dtype=np.float32)


        filtered = cv2.filter2D(
            image_array,
            -1,
            kernel
        )


    # Sobel Edge Filter

    elif filter_name == "Sobel":

        gray = cv2.cvtColor(
            image_array,
            cv2.COLOR_RGB2GRAY
        )


        sobel_x = cv2.Sobel(
            gray,
            cv2.CV_64F,
            1,
            0,
            ksize=3
        )


        sobel_y = cv2.Sobel(
            gray,
            cv2.CV_64F,
            0,
            1,
            ksize=3
        )


        magnitude = cv2.magnitude(
            sobel_x.astype(np.float32),
            sobel_y.astype(np.float32)
        )


        magnitude = cv2.normalize(
            magnitude,
            None,
            0,
            255,
            cv2.NORM_MINMAX
        )


        magnitude = magnitude.astype(
            np.uint8
        )


        filtered = cv2.cvtColor(
            magnitude,
            cv2.COLOR_GRAY2RGB
        )


    else:

        raise ValueError(
            "Unknown filter: " + filter_name
        )


    return Image.fromarray(
        filtered
    )


# Dataset that applies the selected filter

class FilteredDataset(Dataset):

    def __init__(
        self,
        dataset,
        indices,
        filter_name,
        transform
    ):

        self.dataset = dataset

        self.indices = list(
            indices
        )

        self.filter_name = filter_name

        self.transform = transform


    def __len__(self):

        return len(
            self.indices
        )


    def __getitem__(self, index):

        original_index = self.indices[
            index
        ]


        image_path, label = (
            self.dataset.samples[
                original_index
            ]
        )


        image = self.dataset.loader(
            image_path
        )


        image = apply_filter(
            image,
            self.filter_name
        )


        image = self.transform(
            image
        )


        return image, label

In [ ]:
# ============================================================
# 5. TRAINING
# ============================================================

def train_model(
    model,
    train_loader
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )


    loss_function = nn.CrossEntropyLoss()


    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0.0


        for images, labels in train_loader:

            images = images.to(
                DEVICE
            )

            labels = labels.to(
                DEVICE
            )


            optimizer.zero_grad()


            outputs = model(
                images
            )


            loss = loss_function(
                outputs,
                labels
            )


            loss.backward()

            optimizer.step()


            running_loss += loss.item()


        average_loss = (
            running_loss /
            max(1, len(train_loader))
        )


        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"- Loss: {average_loss:.4f}"
        )


    return model

In [ ]:
# ============================================================
# 6. EVALUATION
# ============================================================

def evaluate_model(
    model,
    data_loader
):

    model.eval()


    true_labels = []

    predicted_labels = []

    probabilities = []


    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(
                DEVICE
            )


            outputs = model(
                images
            )


            probs = torch.softmax(
                outputs,
                dim=1
            )


            predictions = probs.argmax(
                dim=1
            )


            true_labels.extend(
                labels.numpy()
            )


            predicted_labels.extend(
                predictions.cpu().numpy()
            )


            probabilities.extend(
                probs.cpu().numpy()
            )


    true_labels = np.array(
        true_labels
    )

    predicted_labels = np.array(
        predicted_labels
    )

    probabilities = np.array(
        probabilities
    )


    # Accuracy

    accuracy = accuracy_score(
        true_labels,
        predicted_labels
    ) * 100


    # Precision

    precision = precision_score(
        true_labels,
        predicted_labels,
        average="weighted",
        zero_division=0
    ) * 100


    # Recall

    recall = recall_score(
        true_labels,
        predicted_labels,
        average="weighted",
        zero_division=0
    ) * 100


    # Weighted F1-score

    weighted_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="weighted",
        zero_division=0
    ) * 100


    # Macro F1-score

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0
    ) * 100


    # AUC

    try:

        auc = roc_auc_score(
            true_labels,
            probabilities,
            multi_class="ovr",
            average="macro"
        ) * 100

    except ValueError:

        auc = np.nan


    return {

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1-score": weighted_f1,

        "Macro-F1": macro_f1,

        "AUC": auc
    }

In [ ]:
# ============================================================
# 8. COMPARATIVE ANALYSIS
# ============================================================

RESULTS = []


TOTAL_EXPERIMENTS = (
    len(MODELS) *
    len(FILTERS)
)


experiment_number = 0


# Run all 18 experiments

for model_name in MODELS:

    for filter_name in FILTERS:

        experiment_number += 1


        print("\n")
        print("=" * 70)

        print(
            f"Experiment "
            f"{experiment_number}/{TOTAL_EXPERIMENTS}"
        )

        print(
            "Model:",
            model_name
        )

        print(
            "Filter:",
            filter_name
        )

        print("=" * 70)


        # Training dataset

        training_dataset = FilteredDataset(
            base_dataset,
            train_indices,
            filter_name,
            train_transform
        )


        # Validation dataset

        validation_dataset = FilteredDataset(
            base_dataset,
            validation_indices,
            filter_name,
            evaluation_transform
        )


        # Training loader

        train_loader = DataLoader(
            training_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        )


        # Validation loader

        validation_loader = DataLoader(
            validation_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        )


        # Test loader

        if test_dataset is not None:

            test_indices = np.arange(
                len(test_dataset)
            )


            evaluation_dataset = FilteredDataset(
                test_dataset,
                test_indices,
                filter_name,
                evaluation_transform
            )


            evaluation_loader = DataLoader(
                evaluation_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=torch.cuda.is_available()
            )

        else:

            evaluation_loader = validation_loader


        # Load model

        model = load_model(
            model_name
        )


        # Train model

        model = train_model(
            model,
            train_loader
        )


        # Evaluate model

        metrics = evaluate_model(
            model,
            evaluation_loader
        )


        # Store results

        result = {

            "Model": model_name,

            "Filter": filter_name,

            **metrics
        }


        RESULTS.append(
            result
        )


        print("\nResults:")

        for key, value in metrics.items():

            if pd.isna(value):

                print(
                    f"{key}: N/A"
                )

            else:

                print(
                    f"{key}: {value:.4f}%"
                )


        # Save model

        output_folder = Path(
            "task02_results"
        )


        model_folder = (
            output_folder /
            "models"
        )


        model_folder.mkdir(
            parents=True,
            exist_ok=True
        )


        safe_model_name = (
            model_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
        )


        safe_filter_name = (
            filter_name
            .lower()
            .replace(" ", "_")
        )


        model_path = (
            model_folder /
            f"{safe_model_name}_{safe_filter_name}.pth"
        )


        torch.save(
            model.state_dict(),
            model_path
        )


        # Free memory

        del model

        del train_loader

        del validation_loader


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# ------------------------------------------------------------
# Create final comparison table
# ------------------------------------------------------------

results_df = pd.DataFrame(
    RESULTS
)


results_df = results_df[
    [
        "Model",
        "Filter",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Macro-F1",
        "AUC"
    ]
]


results_df.iloc[:, 2:] = (
    results_df.iloc[:, 2:].round(4)
)


print("\n")
print("=" * 90)
print("FINAL COMPARISON TABLE")
print("=" * 90)


print(
    results_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Find best results
# ------------------------------------------------------------

best_accuracy = results_df.loc[
    results_df["Accuracy"].idxmax()
]


best_macro_f1 = results_df.loc[
    results_df["Macro-F1"].idxmax()
]


best_auc = results_df.loc[
    results_df["AUC"].idxmax()
]


print("\n")
print("=" * 70)
print("COMPARATIVE ANALYSIS")
print("=" * 70)


print("\nHighest Accuracy:")

print(
    best_accuracy[
        [
            "Model",
            "Filter",
            "Accuracy"
        ]
    ].to_string()
)


print("\nHighest Macro-F1:")

print(
    best_macro_f1[
        [
            "Model",
            "Filter",
            "Macro-F1"
        ]
    ].to_string()
)


print("\nHighest AUC:")

print(
    best_auc[
        [
            "Model",
            "Filter",
            "AUC"
        ]
    ].to_string()
)


# ------------------------------------------------------------
# Create summary
# ------------------------------------------------------------

summary_df = pd.DataFrame([

    [
        "Highest Accuracy",
        best_accuracy["Model"],
        best_accuracy["Filter"],
        best_accuracy["Accuracy"]
    ],

    [
        "Highest Macro-F1",
        best_macro_f1["Model"],
        best_macro_f1["Filter"],
        best_macro_f1["Macro-F1"]
    ],

    [
        "Highest AUC",
        best_auc["Model"],
        best_auc["Filter"],
        best_auc["AUC"]
    ]

], columns=[

    "Measure",
    "Model",
    "Filter",
    "Value (%)"

])


summary_df["Value (%)"] = (
    summary_df["Value (%)"].round(4)
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

output_folder = Path(
    "task02_results"
)


output_folder.mkdir(
    exist_ok=True
)


excel_file = (
    output_folder /
    "Task_02_Filtering_Comparison.xlsx"
)


csv_file = (
    output_folder /
    "Task_02_Filtering_Results.csv"
)


# Save Excel

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Comparison Table",
        index=False
    )


    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )


# Save CSV

results_df.to_csv(
    csv_file,
    index=False
)


# ------------------------------------------------------------
# Final message
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("TASK 02 COMPLETED")
print("=" * 70)


print("\nExcel file:")
print(excel_file)


print("\nCSV file:")
print(csv_file)


print("\nSaved models:")
print(
    output_folder / "models"
)



Experiment 1/18
Model: EfficientNet-B0
Filter: No Filter
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 133MB/s]


Epoch 1/5 - Loss: 1.8545
Epoch 2/5 - Loss: 1.2658
Epoch 3/5 - Loss: 0.9673
Epoch 4/5 - Loss: 0.7809
Epoch 5/5 - Loss: 0.6785

Results:
Accuracy: 62.7119%
Precision: 63.2996%
Recall: 62.7119%
F1-score: 59.9575%
Macro-F1: 58.1597%
AUC: 93.7319%


Experiment 2/18
Model: EfficientNet-B0
Filter: Average
Epoch 1/5 - Loss: 1.8166
Epoch 2/5 - Loss: 1.1964
Epoch 3/5 - Loss: 0.9158
Epoch 4/5 - Loss: 0.7530
Epoch 5/5 - Loss: 0.6582

Results:
Accuracy: 58.4746%
Precision: 64.3430%
Recall: 58.4746%
F1-score: 55.4163%
Macro-F1: 54.4383%
AUC: 91.1231%


Experiment 3/18
Model: EfficientNet-B0
Filter: Gaussian
Epoch 1/5 - Loss: 1.8224
Epoch 2/5 - Loss: 1.2112
Epoch 3/5 - Loss: 0.9406
Epoch 4/5 - Loss: 0.7745
Epoch 5/5 - Loss: 0.6237

Results:
Accuracy: 51.6949%
Precision: 53.1601%
Recall: 51.6949%
F1-score: 46.4985%
Macro-F1: 44.8737%
AUC: 90.5551%


Experiment 4/18
Model: EfficientNet-B0
Filter: Median
Epoch 1/5 - Loss: 1.8205
Epoch 2/5 - Loss: 1.1973
Epoch 3/5 - Loss: 0.8963
Epoch 4/5 - Loss: 0.7780


100%|██████████| 30.8M/30.8M [00:00<00:00, 179MB/s]


Epoch 1/5 - Loss: 1.5230
Epoch 2/5 - Loss: 0.9123
Epoch 3/5 - Loss: 0.6964
Epoch 4/5 - Loss: 0.5381
Epoch 5/5 - Loss: 0.4605

Results:
Accuracy: 61.0169%
Precision: 63.5072%
Recall: 61.0169%
F1-score: 58.2461%
Macro-F1: 56.7572%
AUC: 91.5365%


Experiment 8/18
Model: DenseNet121
Filter: Average
Epoch 1/5 - Loss: 1.5164
Epoch 2/5 - Loss: 0.8798
Epoch 3/5 - Loss: 0.6828
Epoch 4/5 - Loss: 0.5258
Epoch 5/5 - Loss: 0.4405

Results:
Accuracy: 55.0847%
Precision: 57.7177%
Recall: 55.0847%
F1-score: 52.2379%
Macro-F1: 50.5442%
AUC: 89.1872%


Experiment 9/18
Model: DenseNet121
Filter: Gaussian
Epoch 1/5 - Loss: 1.5058
Epoch 2/5 - Loss: 0.8949
Epoch 3/5 - Loss: 0.6880
Epoch 4/5 - Loss: 0.5626
Epoch 5/5 - Loss: 0.4522

Results:
Accuracy: 55.0847%
Precision: 57.8127%
Recall: 55.0847%
F1-score: 51.6417%
Macro-F1: 50.0556%
AUC: 88.9693%


Experiment 10/18
Model: DenseNet121
Filter: Median
Epoch 1/5 - Loss: 1.4701
Epoch 2/5 - Loss: 0.8656
Epoch 3/5 - Loss: 0.6701
Epoch 4/5 - Loss: 0.5467
Epoch 5/5 -

100%|██████████| 171M/171M [00:01<00:00, 145MB/s]


Epoch 1/5 - Loss: 1.7433
Epoch 2/5 - Loss: 1.0319
Epoch 3/5 - Loss: 0.6846
Epoch 4/5 - Loss: 0.5118
Epoch 5/5 - Loss: 0.4027

Results:
Accuracy: 59.3220%
Precision: 63.4078%
Recall: 59.3220%
F1-score: 57.4111%
Macro-F1: 56.0730%
AUC: 91.1700%


Experiment 14/18
Model: ResNet101
Filter: Average
Epoch 1/5 - Loss: 1.7670
Epoch 2/5 - Loss: 1.0995
Epoch 3/5 - Loss: 0.8013
Epoch 4/5 - Loss: 0.5714
Epoch 5/5 - Loss: 0.4567

Results:
Accuracy: 55.0847%
Precision: 63.3236%
Recall: 55.0847%
F1-score: 54.8831%
Macro-F1: 51.7445%
AUC: 88.0733%


Experiment 15/18
Model: ResNet101
Filter: Gaussian
Epoch 1/5 - Loss: 1.7647
Epoch 2/5 - Loss: 1.0875
Epoch 3/5 - Loss: 0.7297
Epoch 4/5 - Loss: 0.5191
Epoch 5/5 - Loss: 0.4167

Results:
Accuracy: 55.0847%
Precision: 61.8857%
Recall: 55.0847%
F1-score: 53.1033%
Macro-F1: 51.2533%
AUC: 89.1351%


Experiment 16/18
Model: ResNet101
Filter: Median
Epoch 1/5 - Loss: 1.7672
Epoch 2/5 - Loss: 1.0934
Epoch 3/5 - Loss: 0.7221
Epoch 4/5 - Loss: 0.5355
Epoch 5/5 - Los